Устанавливаем все нужные библиотеки

In [1]:
%pip install python-docx

   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   --------------- ------------------------ 1.6/4.0 MB 10.0 MB/s eta 0:00:01
   ------------------------------------ --- 3.7/4.0 MB 10.2 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 10.1 MB/s  0:00:00

   -------------------- ------------------- 1/2 [python-docx]
   ---------------------------------------- 2/2 [python-docx]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Все документы которые у нас есть нужно перевести из doxc в json формат для дальнейшей работы. Импортируем нужные библиотеки для работы с ос, расширениями файлов и тд,а так же из докс импортируем класс документ для работы с расширением, затем задем путь где у нас лежат документы и как будет называться наш конечный файл json (r нужна для того что бы питон корректно воспринимал \ в пути), теперь нам нужно почистить наши документы (с этим еще можно поработать), я решил заменить упоминание "федерального, закон" во всех вариациях на пустоту,заменить все пробелы,переносы строк и тд на обычный один пробел, затем наша функция возвращает почищеный текст. Теперь создадим функцию извлечения текста doc = Document(file_path) открывает и загружает в память наш файл, затем создаем список куда мы будем складывать наши абзацы,создаем цикл который итеративно перебирает абзацы, берем наш сырой текст абзаца и прогоняем его через очистку, так же смотрим не получилась ли у нас пустая строчка и если все впорядке то добавляем абзац в список, в возврате мы берем все что у нас получилось и склеиваем вставляя между ними перенос строки. Теперь функция для нашего json файла, создаем список который в конце будет нашим файлом, читаем название всех наших файлов которые в нашей папке data, смотрим что бы все было в формате .docx, склеиваем путь к папке и имя файла, отдаем файл нашей функции и получаем готовый текст, !!сохраняем словарь вместе с текстом и ключом ввиде файла из которого был взят текст,это нужно для того что бы LLM в будущем при цитировании могла точно сказать откуда она взяла тот или иной текст. Теперь открываем файл для записи (указывая 0encoding='utf-8' для адекватной работы русского языка), выгружаем сисок словарей в json файл, (if __name__ == "__main__":) для ннашего запуска всей функции 

In [ ]:
import os
import json
import re
from docx import Document

DATA_DIR = r"C:\Users\asus\Desktop\LLM law\data" 
OUTPUT_FILE = "parsed_documents.json"

def clean_text(text):
   
    text = re.sub(r'\(в ред\. Федеральны[хго]+ закон[аов]+.*?\)', '', text)
    
   
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def parse_docx(file_path):
    doc = Document(file_path)
    full_text = []
    
    for para in doc.paragraphs:
        cleaned_para = clean_text(para.text)
        if cleaned_para:
            full_text.append(cleaned_para)
            
    return "\n".join(full_text)

def main():
    documents = []
    
    
    for filename in os.listdir(DATA_DIR):
        if filename.endswith(".docx"):
            file_path = os.path.join(DATA_DIR, filename)
            print(f"Парсинг файла: {filename}...")
            
            text_content = parse_docx(file_path)
            
            documents.append({
                "source": filename,
                "content": text_content
            })
            
    
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(documents, f, ensure_ascii=False, indent=4)
        
    print(f"Готово! Обработано файлов: {len(documents)}. Результат сохранен в {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

Парсинг файла: Государственная информационная система мониторинга в сфере межнациональных и межконфессиональных отношений и раннего предупреждения конфликтных ситуаций.docx...
Парсинг файла: Закон Российской Федерации от 21.07.1993 №5485-1.docx...
Парсинг файла: Приказ ФАДН России от 23.10.2015 №70 (ред. от 08.04.2022, с изм. от 22.06.2023).docx...
Парсинг файла: Федеральный закон от 12.01.1996 №7-ФЗ (ред. от 26.02.2024).docx...
Парсинг файла: Федеральный закон от 24.05.1999 №99-ФЗ (ред. от 23.07.2013).docx...
Готово! Обработано файлов: 5. Результат сохранен в parsed_documents.json


Теперь нам нужно проверить все ли нормально у нас перевелось, имопртируем библиотеку для чтения json и для рандомной генерации (взять какой нибудь текст что бы понять нормально ли все там), создаем функцию в которой мы открываем наш файл в режиме чтения,берем текст и превращаем его обратно в список словарей для python, считаем сколько у нас всего получилось документов, затем заводим счетчик для того что бы понимать длину каждого текста и в конце понять сколько у нас получилось символов, заводим цикл с порядковым номером, вытаскиваем сам файл,его текст и количество символов добавляя их в общую копилку, выводим номер документа и количество символов в нем, а так же общее их количество, теперь мы берем рандомный документ, указываем что бы текст брался из середины для большего понимания текста, затем от этой середины мы берем 1000 символом и их уже выводим

In [16]:
import json
import random

INPUT_FILE = "parsed_documents.json"

def check_parsed_data():
    
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        documents = json.load(f)
        
    print(f"ОБЩАЯ СТАТИСТИКА:")
    print(f"Всего документов в базе: {len(documents)}\n")
    
    total_chars = 0
    for i, doc in enumerate(documents):
        source = doc['source']
        text = doc['content']
        chars_count = len(text)
        total_chars += chars_count
        print(f"[{i+1}] {source}")
        print(f"    Длина текста: {chars_count:,} символов")
    
    print(f"\nОбщий объем всей базы: {total_chars:,} символов")
    
    random_doc = random.choice(documents)
    print(f"\nслучайный фрагмент текста из: {random_doc['source']}\n")
    
    start_idx = len(random_doc['content']) // 2
    snippet = random_doc['content'][start_idx : start_idx + 1000]
    
    print(snippet)

check_parsed_data()

ОБЩАЯ СТАТИСТИКА:
Всего документов в базе: 5

[1] Государственная информационная система мониторинга в сфере межнациональных и межконфессиональных отношений и раннего предупреждения конфликтных ситуаций.docx
    Длина текста: 3,250 символов
[2] Закон Российской Федерации от 21.07.1993 №5485-1.docx
    Длина текста: 59,304 символов
[3] Приказ ФАДН России от 23.10.2015 №70 (ред. от 08.04.2022, с изм. от 22.06.2023).docx
    Длина текста: 7,714 символов
[4] Федеральный закон от 12.01.1996 №7-ФЗ (ред. от 26.02.2024).docx
    Длина текста: 199,597 символов
[5] Федеральный закон от 24.05.1999 №99-ФЗ (ред. от 23.07.2013).docx
    Длина текста: 34,147 символов

Общий объем всей базы: 304,012 символов

случайный фрагмент текста из: Государственная информационная система мониторинга в сфере межнациональных и межконфессиональных отношений и раннего предупреждения конфликтных ситуаций.docx

изированная обработка данных о событиях, связанных с межнациональными и межконфессиональными отношениями (де